In [107]:
from dotenv import load_dotenv
from os.path import join, dirname
from sklearn.preprocessing   import OneHotEncoder
from nltk.tokenize import  RegexpTokenizer
import numpy as np
import os
import json

In [108]:
# Obtain our default environment
path_to_env = os.path.join('..','.env')
print(load_dotenv(path_to_env))
default_path = os.environ['DEFAULT_PATH']
print(default_path)

True
/home/drew/Split-Learning/


### Obtain the directoties of the training and testing set within mullenbachs repo

In [109]:
caml_mimic_path  = os.path.join(default_path,'caml-mimic')
mimic_data_path  = os.path.join(caml_mimic_path,'mimicdata')
mimic3_data_path = os.path.join(mimic_data_path,'mimic3')

train_50_path = os.path.join(mimic3_data_path,'train_50.csv')
test_50_path  = os.path.join(mimic3_data_path,'test_50.csv')

### Obtain our working directory

In [110]:
data_path = os.path.join(default_path,'Data')
mullen_parsed = os.path.join(data_path,'MullenBach_Parsed')

### Load in our datasets

In [111]:
import pandas as pd

In [112]:
train_50 = pd.read_csv(train_50_path,delimiter=',')
print(len(train_50['TEXT'].iloc[0].split()))
train_50.head(n = 3)

105


,SUBJECT_ID,HADM_ID,TEXT,LABELS,length
0,7908,182396,admission date discharge date date of birth se...,287.5;584.9;45.13,105
1,11231,183363,admission date discharge date date of birth se...,96.71;272.4;401.9,106
2,3184,144347,admission date discharge date date of birth se...,530.81,117


In [113]:
test_50 = pd.read_csv(test_50_path,delimiter=',')
test_50.head(n = 3)

,SUBJECT_ID,HADM_ID,TEXT,LABELS,length
0,92003,193800,admission date discharge date date of birth se...,96.71;96.04;518.81,235
1,95088,158927,admission date discharge date date of birth se...,V45.81;96.71;401.9,369
2,96937,129034,admission date discharge date date of birth se...,427.31;96.71;038.9;995.92;250.00,370


### Need to reformat the labels to be one hot encoding and save the translation

In [114]:
all_labels = list({l for row in train_50['LABELS'] for l in row.split(';')})
len(all_labels)

50

In [115]:
enc = OneHotEncoder().fit(np.array(all_labels).reshape(-1,1)); enc

OneHotEncoder()

In [116]:
sample_row = train_50['LABELS'].iloc[0]
final_label = np.zeros(shape = (1,50)) # (1, label_space)
for label in sample_row.split(';'):
    label = np.array(label).reshape(1, -1)
    label = enc.transform(label).A
    final_label += label
final_label.flatten()

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

----

### Encoding Tokens

In [117]:
current_dir    = os.path.join(default_path,'Replicating Mullenbach')
vocab_map_path = os.path.join(current_dir,'Vocab_Mappings.json')

tokenizer = RegexpTokenizer(r'\w+')

In [118]:
all_tokens_train = {t for entry in train_50['TEXT'] for t in tokenizer.tokenize(entry)}
all_tokens_test  = {t for entry in test_50['TEXT'] for t in tokenizer.tokenize(entry)}
print(len(all_tokens_test))
print(len(all_tokens_train))

all_tokens = all_tokens_test | all_tokens_train

31999
59167


In [119]:
mapping = {idx:v for v,idx in enumerate(all_tokens)}
with open(vocab_map_path, 'w') as f:
    json.dump(mapping, f)

### With the word mappings and label mapping, we can construct our datasets

In [120]:
train_dataset = pd.DataFrame(columns=['TEXT','LABEL'])
test_dataset  = pd.DataFrame(columns=['TEXT','LABEL'])

In [121]:
text_hot_enc = lambda row: [mapping[token] for token in tokenizer.tokenize(row)]
train_dataset['TEXT'] = train_50['TEXT'].apply(func = text_hot_enc)
test_dataset['TEXT']  =  test_50['TEXT'].apply(func = text_hot_enc)
train_dataset.head(n = 3)

,TEXT,LABEL
0,"[40606, 7074, 36455, 7074, 7074, 7615, 4291, 2...",NaN
1,"[40606, 7074, 36455, 7074, 7074, 7615, 4291, 2...",NaN
2,"[40606, 7074, 36455, 7074, 7074, 7615, 4291, 2...",NaN


### Encode our labels from before

In [122]:
def encode_onehot_label(row):
    final_label = np.zeros(shape = (1,50)) # (1, label_space)
    for label in row.split(';'):
        label = np.array(label).reshape(1, -1)
        label = enc.transform(label).A
        final_label += label
    return final_label.flatten()

train_dataset['LABEL'] = train_50['LABELS'].apply(func = encode_onehot_label)
test_dataset['LABEL']  =  test_50['LABELS'].apply(func = encode_onehot_label)
train_dataset.head(n = 3)

,TEXT,LABEL
0,"[40606, 7074, 36455, 7074, 7074, 7615, 4291, 2...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
1,"[40606, 7074, 36455, 7074, 7074, 7615, 4291, 2...","[0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, ..."
2,"[40606, 7074, 36455, 7074, 7074, 7615, 4291, 2...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [123]:
sample = train_dataset['TEXT'].iloc[0]
sample = np.array(sample)

In [124]:
MIN_TOKEN_LEN = 2500 #max([len(entry) for entry in train_dataset['TEXT']]) # -> 7567
def padding_trunc(row):
    MIN_TOKEN_LEN = 2500
    row = np.array(row)
    # If its smaller we need to truncate
    if len(row) > MIN_TOKEN_LEN:
        return row[:MIN_TOKEN_LEN]
    # Otherwise we pad it with zeroes
    return np.pad(row, pad_width=(0, MIN_TOKEN_LEN - len(row)), mode='constant')

train_dataset['TEXT'] = train_dataset['TEXT'].apply(func = padding_trunc)
test_dataset['TEXT']  =  test_dataset['TEXT'].apply(func = padding_trunc)

In [125]:
print(max([len(entry) for entry in train_dataset['TEXT']]) )# -> 7567
print(min([len(entry) for entry in train_dataset['TEXT']]) )# -> 7567
print(max([len(entry) for entry in test_dataset['TEXT']]) )
print(min([len(entry) for entry in test_dataset['TEXT']]) )

2500
2500
2500
2500


### Save our datasets to our current directory

In [131]:
train_dataset['TEXT']  = train_dataset['TEXT'] .apply(func = list)
train_dataset['LABEL'] = train_dataset['LABEL'].apply(func = list)

test_dataset['TEXT']  = test_dataset['TEXT'] .apply(func = list)
test_dataset['LABEL'] = test_dataset['LABEL'].apply(func = list)

In [140]:
#train_dataset.dtypes = {'TEXT':  list, 'LABEL': list}

In [ ]:
train_csv_path = os.path.join(current_dir,'train_50.csv')
test_csv_path  = os.path.join(current_dir,'test_50.csv')

test_dataset .to_csv(test_csv_path  ,index=False)
train_dataset.to_csv(train_csv_path ,index=False)